In [0]:
import json
from pathlib import Path
import pandas as pd

In [0]:

dbutils.widgets.text("raw_root", "/Volumes/workspace/raw/tvmze", "Raw Root")
# Léalos en Python
raw_root = Path(dbutils.widgets.get("raw_root"))

In [0]:
%sql
DROP DATABASE IF EXISTS workspace.bronze CASCADE; 

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bronze
COMMENT 'Capa Bronze: datos crudos procesados'

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.bronze.predios_registro")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.predios_registro (
    pk STRING,
    matricula STRING,
    fecha_radica_texto STRING,
    fecha_apertura_texto STRING,
    year_radica STRING,
    orip STRING,
    divipola STRING,
    departamento STRING,
    municipio STRING,
    tipo_predio_zona STRING,
    categoria_ruralidad_2024 STRING,
    num_anotacion INT,
    estado_folio STRING,
    folios_derivados STRING,
    dinamica_2024 INT,
    cod_natujur INT,
    nombre_natujur STRING,
    numero_catastral_antiguo STRING,
    documento_justificativo STRING,
    count_a INT,
    count_de INT,
    predios_nuevos INT,
    tiene_valor INT,
    tiene_mas_de_un_valor INT,
    valor DOUBLE
)
USING DELTA
""")

In [0]:
candidate_files = sorted(raw_root.glob("**/tvmaze.json"))

records: list[dict] = []
for json_file in candidate_files:
    payload = json.loads(json_file.read_text(encoding="utf-8"))
    records.extend(payload)  # asumimos que payload siempre es lista

df = pd.json_normalize(records) if records else pd.DataFrame()
df_spark = spark.createDataFrame(df)

from pyspark.sql.functions import lit

expected_cols = [
    "pk",
    "matricula",
    "numero_catastral",
    "categoria_ruralidad_2024",
    "dinamica_2024",
    "valor"
]

for col in expected_cols:
    if col not in df_spark.columns:
        df_spark = df_spark.withColumn(col, lit(None))

In [0]:
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql import functions as F

# Castear columnas a INT según el esquema de la tabla
int_columns = ['num_anotacion', 'dinamica_2024', 'cod_natujur', 'count_a', 'count_de', 
               'predios_nuevos', 'tiene_valor', 'tiene_mas_de_un_valor']
for col_name in int_columns:
    if col_name in df_spark.columns:
        df_spark = df_spark.withColumn(col_name, F.col(col_name).cast(IntegerType()))

# Castear columna valor a DOUBLE
if 'valor' in df_spark.columns:
    df_spark = df_spark.withColumn('valor', F.col('valor').cast(DoubleType()))


# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.bronze.predios_registro")

In [0]:
df.columns